In [0]:
%sql
show catalogs;

In [0]:
%sql
show schemas in handson;

In [0]:
show tables in handson.testschema

create one employees table in the testschema in handson catalog


In [0]:

CREATE TABLE handson.testschema.employees
   (name STRING, dept STRING, salary INT, age INT);

In [0]:
USE CATALOG handson;
USE SCHEMA testschema;
SELECT current_schema();

In [0]:
insert into employees
values ('Lisa', 'Sales', 10000, 35),
          ('Evan', 'Sales', 32000, 38),
          ('Fred', 'Engineering', 21000, 28),
          ('Alex', 'Sales', 30000, 33),
          ('Tom', 'Engineering', 23000, 33),
          ('Jane', 'Marketing', 29000, 28),
          ('Jeff', 'Marketing', 35000, 38),
          ('Paul', 'Engineering', 29000, 23),
          ('Chloe', 'Engineering', 23000, 25);

In [0]:
select * from employees
limit 5;

## now lets use window functions on this dataset

1. Rank() 

In [0]:
with cte_employees as(
    select *,
    rank() over(partition by dept order by salary desc)
    as rank
    from employees
)

select * from cte_employees
limit 5;

2. row_number()

In [0]:
with cte_employees as(
    select *,
    row_number() over(partition by dept order by salary desc)
    as rank
    from employees
)

select * from cte_employees
limit 5;

3. dense_rank()

In [0]:
with cte_employees as(
    select *,
    dense_rank() over(partition by dept order by salary desc)
    as rank
    from employees
)

select * from cte_employees
limit 5;

4. CUME_DIST(),


 what if we what to find how many employees are less then or equal to the current employee salary kind of 

In [0]:
with cte_employees as(
    select *,
    cume_dist() over(partition by dept order by salary desc)
    as rank
    from employees
)

select * from cte_employees
limit 5;

5. LAG() - return the value of the prev row if it exit otherwise return NULL
6. LEAD() _ return the value of the next row if it exit otherwise reutn 0

In [0]:
with cte_employees as(
    select *,
    lag(age) over(partition by dept order by salary )
    as laging,
    lead(salary) over(partition by dept order by salary )
    as leading
    from employees
)

select * from cte_employees
limit 5;

# UDF
```python
pyspark.sql.functions.udf(f=None, returnType=StringType(), *, useArrow=None)
# this  Creates a user defined function (UDF).
```


* Parameters :
- f : function, optional : python function if used as a standalone function
- returnType : pyspark.sql.types.DataType or str, optional
the return type of the user-defined function. The value can be either a pyspark.sql.types.DataType object or a DDL-formatted type string. Defaults to StringType.
- useArrow  : bool, optional : whether to use Arrow to optimize the (de)serialization. When it is None, the Spark config “spark.sql.execution.pythonUDF.arrow.enabled” takes effect.



## UDF Deterministic Behavior

* PySpark **UDFs are deterministic by default**.
* **Deterministic function:** Same input always produces the same output.
* Spark assumes that a deterministic UDF will return the same result for the same input.
* Because of Spark's **query optimization**, duplicate UDF calls may be eliminated or their results may be reused.
* In some cases, Spark may also invoke the UDF **more times than it appears in the query**.
* If a UDF can produce different results for the same input, it is **non-deterministic**.
* Examples of non-deterministic operations include:

  * Random number generation
  * Current timestamp/time-based values
  * Other operations whose output can change for the same input
* For such functions, the UDF should be explicitly marked as **non-deterministic** using `asNondeterministic()`.
* Avoid putting **side effects** such as file writes, API calls, or external state changes inside UDFs because Spark may optimize or re-evaluate them.


In [0]:
%python
from pyspark.sql import functions as F
from pyspark.sql import types as T
import random
random_udf = F.udf(lambda: F.current_date(), T.IntegerType()).asNondeterministic()





In [0]:
%sql 
select current_schema()

In [0]:
show tables;

In [0]:
%python
df = spark.table("employees")

In [0]:
%python
# now lets apply the udf function on it from


from datetime import date
def generate_random():    
    return str(date.today())

random_udf = F.udf(
    generate_random,
    T.StringType()
)

df.select(
    "*",
    random_udf().alias("random_number")
).show(3)


In [0]:
%python 
# for best practices we need to use  decorators in the udf like 

@udf(returnType=T.StringType())
def processing_date(Salary):
    if Salary > 50000:
        return "High"
    else:
        return "Low"

df.select( "*", processing_date(df['Salary']).alias("pay-class") ).show(3)
